# Day 09：面向对象编程（下）（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 2 周**：第 2 周：面向对象、工程化与命令行 AI 助手实战  
> **GitLab 工单**：`STARMIND-009` ｜ **开发分支**：`feat/day09-oop-model-adapter`  
> **配套 HTML5 课件**：`courseware/day09/index.html` ｜ **独立版本快照**：`versions/day09/`

---

## 🎙️ 老陈开场旁白

昨天刚说完封装，今天一早 CTO 在群里下发了一个硬指标：星辰智脑不能被任何一家大模型厂商绑死！业务层写代码时，只管调 `model.generate(prompt)`，底层今天想切 DeepSeek/OpenAI、明天想切通义千问 Qwen，业务代码一行都不许改！这正是面向对象最迷人的杀手锏——**继承、抽象基类与多态**！今天 STARMIND-009 咱们亲手设计 `BaseModel -> OpenAIModel / QwenModel` 统一适配层，再配上 `@property` 管住 `temperature` 合法范围，加上 `__str__`、`__len__`、`__call__` 魔术方法，写完你就彻底懂什么叫大厂架构感了。

### 承上启下
- **昨日基线**：昨天咱们封装了 ChatMessage 与 ChatSession 领域对象，解决了裸字典无校验的痛点。
- **明日预告**：明天（Day 10）咱们把单文件脚本拆分成多文件工程包（Package），引入自定义异常体系与 venv 依赖管理。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day09'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-009）

企业 AI 平台需同时接入多家大模型供应商以实现高可用容灾与成本优化，但各家接口细节与计费参数不同，需要统一的抽象适配层屏蔽差异。

**今日迭代目标：**
- 掌握继承、`super()`、抽象基类（`abc.ABC` / `@abstractmethod`）与多态分发。
- 熟练使用 `@property` 与 `@xxx.setter` 实现带范围校验的安全属性。
- 理解高频魔术方法 `__str__`、`__repr__`、`__len__`、`__call__` 的工程用法。

**验收标准：**
- 定义 `BaseModel` 抽象基类，禁止直接实例化，强制子类实现 `generate()` 方法。
- 实现 `OpenAIModel` 与 `QwenModel` 两个子类，重写各自请求头构造与本地模拟生成逻辑。
- 通过 `@property` 限制 `temperature` 必须在 `[0.0, 2.0]` 之间，越界抛 `ValueError`；pytest 全绿。

---

## 📚 课堂核心技术笔记

### 1. 1. 抽象基类（ABC）与 super() 继承

继承 `abc.ABC` 并用 `@abstractmethod` 装饰的方法，相当于给所有子类签下「军令状」：子类必须实现该方法，否则无法实例化。子类重写 `__init__` 时，务必第一行调用 `super().__init__(...)`，确保父类初始化的属性全部就绪。

> **⚠️ 老陈避坑**：如果父类定义了 `self.temperature = temperature` 走的是 `@property` setter，那么子类通过 `super().__init__` 传入非法温度值时也会自动触发校验，一处编写全族受益！

### 2. 2. @property 让方法像属性一样自然读写

相比 Java 满屏的 `getTemperature()` / `setTemperature()`，Python 用 `@property` 和 `@temperature.setter` 保持 `obj.temperature` 的简洁语法，同时在背后悄悄完成范围校验或延迟计算。

> **⚠️ 老陈避坑**：在 setter 里千万不要写 `self.temperature = value`（会无限递归自己调用自己导致 `RecursionError`），必须写入带下划线的底层属性 `self._temperature = value`！

### 3. 3. 常用魔术方法（Dunder Methods）

- `__str__`：面向用户的友好打印（`print(obj)`）；`__repr__`：面向开发者的精确调试展示。
- `__len__`：支持 `len(router)` 查看注册了多少个模型。
- `__call__`：让对象实例可以像函数一样被括号调用 `model('你好')`，PyTorch 的 `nn.Module` 和 LangChain 的 `Runnable` 都在用！

> **⚠️ 老陈避坑**：重写 `__eq__` 后如果还想把对象放进 `set` 或当作 `dict` 键，必须同时实现 `__hash__`。


## 💻 实操步骤 1：步骤一：定义星辰智脑抽象基类 BaseModel（含 @property 与魔术方法）

- **目标文件**：`versions/day09/src/day09_base_model.py`
- **老陈代码旁白**：先立规矩！定义抽象基类 `BaseModel`，管好 `temperature` 合法范围、统一调用计数与 `__call__` 便捷入口。


In [ ]:
"""
STARMIND-009 模块一：大模型统一抽象基类 BaseModel。
演示 abc.ABC 抽象基类、@property 范围校验、模板方法与 __call__ / __str__ 魔术方法。
"""

from abc import ABC, abstractmethod
from typing import Dict, Any


class BaseModel(ABC):
    """星辰智脑所有大模型适配器的抽象基类。"""

    provider: str = "base"

    def __init__(self, model_name: str, temperature: float = 0.7) -> None:
        self.model_name: str = model_name
        # 通过 @temperature.setter 触发构造期范围校验
        self.temperature = temperature
        self._call_count: int = 0

    @property
    def temperature(self) -> float:
        """读取采样温度（只通过受控接口暴露私有字段 _temperature）。"""
        return self._temperature

    @temperature.setter
    def temperature(self, value: float) -> None:
        """拦截写入：强制限制温度必须在 [0.0, 2.0] 合法区间内。"""
        val = float(value)
        if not (0.0 <= val <= 2.0):
            raise ValueError(f"temperature 必须在 [0.0, 2.0] 之间，当前传入: {val}")
        self._temperature = val

    @property
    def call_count(self) -> int:
        """只读属性：返回当前模型实例累计被调用的次数。"""
        return self._call_count

    @abstractmethod
    def build_headers(self) -> Dict[str, str]:
        """抽象方法：子类必须实现各自厂商的 HTTP 鉴权请求头。"""

    @abstractmethod
    def _do_generate(self, prompt: str) -> str:
        """抽象方法：子类实现具体的文本生成逻辑（本地模拟或真实 API）。"""

    def generate(self, prompt: str) -> Dict[str, Any]:
        """模板方法：统一完成空值检查、计数统计并委托子类生成回复。"""
        cleaned = prompt.strip()
        if not cleaned:
            raise ValueError("提示词 prompt 不能为空")
        self._call_count += 1
        reply_text = self._do_generate(cleaned)
        return {
            "provider": self.provider,
            "model": self.model_name,
            "temperature": self.temperature,
            "reply": reply_text,
            "call_index": self._call_count,
        }

    def __call__(self, prompt: str) -> Dict[str, Any]:
        """魔术方法：支持直接将模型实例当作函数调用 model('你好')。"""
        return self.generate(prompt)

    def __str__(self) -> str:
        return f"[{self.provider}] {self.model_name} (temp={self.temperature}, calls={self._call_count})"


if __name__ == "__main__":
    print("BaseModel 抽象基类定义完成，禁止直接实例化，请继承后使用。")


### 🔍 步骤 1 代码深度解析

1. `generate()` 是经典的「模板方法」：把通用的空串校验、调用次数自增、结果字典包装留在父类，只把有差异的 `_do_generate()` 交给子类实现。
2. `call_count` 只定义了 `@property` 没写 setter，天然变成了只读属性，外部无法伪造调用次数。

**预期运行输出参考：**
```text
BaseModel 抽象基类定义完成，禁止直接实例化，请继承后使用。
```


## 💻 实操步骤 2：步骤二：实现 OpenAIModel / QwenModel 多态子类与 ModelRouter

- **目标文件**：`versions/day09/src/day09_model_providers.py`
- **老陈代码旁白**：现在让 `OpenAIModel` 和 `QwenModel` 继承 `BaseModel`，再写一个支持 `len(router)` 的 `ModelRouter`，体验一行代码无痛切换模型的丝滑感。


In [ ]:
"""
STARMIND-009 模块二：OpenAIModel / QwenModel 多态实现与多模型路由器。
演示继承、super() 调用、方法重写、多态分发以及 __len__ / __contains__ 魔术方法。
"""

from typing import Dict, Any
from day09_base_model import BaseModel


class OpenAIModel(BaseModel):
    """OpenAI / DeepSeek 兼容协议适配器（本地离线模拟版）。"""

    provider: str = "openai"

    def __init__(self, model_name: str = "gpt-4o-mini", temperature: float = 0.7, api_key: str = "mock-openai-key") -> None:
        super().__init__(model_name=model_name, temperature=temperature)
        self._api_key = api_key

    def build_headers(self) -> Dict[str, str]:
        return {
            "Authorization": f"Bearer {self._api_key}",
            "Content-Type": "application/json",
        }

    def _do_generate(self, prompt: str) -> str:
        return f"[OpenAI:{self.model_name}|t={self.temperature}] 已收到指令「{prompt}」，建议采用结构化分步解答。"


class QwenModel(BaseModel):
    """阿里云通义千问（DashScope）适配器（本地离线模拟版）。"""

    provider: str = "qwen"

    def __init__(self, model_name: str = "qwen-plus", temperature: float = 0.5, api_key: str = "mock-qwen-key") -> None:
        super().__init__(model_name=model_name, temperature=temperature)
        self._api_key = api_key

    def build_headers(self) -> Dict[str, str]:
        return {
            "Authorization": f"Bearer {self._api_key}",
            "X-DashScope-SSE": "disable",
            "Content-Type": "application/json",
        }

    def _do_generate(self, prompt: str) -> str:
        return f"[通义千问:{self.model_name}|t={self.temperature}] 您好！针对「{prompt}」，星辰智脑中文知识库已为您整理完毕。"


class ModelRouter:
    """多模型统一路由器：面向 BaseModel 抽象接口编程，屏蔽厂商差异。"""

    def __init__(self) -> None:
        self._registry: Dict[str, BaseModel] = {}

    def register(self, alias: str, model: BaseModel) -> None:
        if not isinstance(model, BaseModel):
            raise TypeError("注册对象必须是 BaseModel 的子类实例")
        self._registry[alias] = model

    def ask(self, alias: str, prompt: str) -> Dict[str, Any]:
        if alias not in self._registry:
            raise KeyError(f"未注册的模型别名: {alias}")
        model = self._registry[alias]
        # 多态调用：触发 BaseModel.__call__ -> 具体子类的 _do_generate
        return model(prompt)

    def __len__(self) -> int:
        return len(self._registry)

    def __contains__(self, alias: str) -> bool:
        return alias in self._registry


if __name__ == "__main__":
    router = ModelRouter()
    router.register("fast", OpenAIModel("gpt-4o-mini", temperature=0.3))
    router.register("cn_pro", QwenModel("qwen-plus", temperature=0.5))

    print(f"已注册模型数量: {len(router)} | 包含 cn_pro: {'cn_pro' in router}")
    for alias in ("fast", "cn_pro"):
        res = router.ask(alias, "总结一下杭州星辰数科的核心优势")
        print(f"-> {alias}: {res['reply']}")


### 🔍 步骤 2 代码深度解析

1. `ModelRouter.register` 用 `isinstance(model, BaseModel)` 守住类型边界，确保只有遵守契约的模型才能注册进来。
2. 实现 `__len__` 和 `__contains__` 后，路由器原生支持 `len(router)` 和 `'cn_pro' in router`，Pythonic 十足。

**预期运行输出参考：**
```text
已注册模型数量: 2 | 包含 cn_pro: True
-> fast: [OpenAI:gpt-4o-mini|t=0.3] 已收到指令「总结一下杭州星辰数科的核心优势」，建议采用结构化分步解答。
-> cn_pro: [通义千问:qwen-plus|t=0.5] 您好！针对「总结一下杭州星辰数科的核心优势」，星辰智脑中文知识库已为您整理完毕。
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day09/tests/test_day09.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-009 自动化单元测试。"""

import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day09_base_model import BaseModel
from day09_model_providers import OpenAIModel, QwenModel, ModelRouter


def test_cannot_instantiate_abstract_base_model():
    with pytest.raises(TypeError):
        BaseModel("abstract-only")  # type: ignore


def test_temperature_property_validation():
    m = OpenAIModel("gpt-4o-mini", temperature=0.8)
    assert m.temperature == 0.8
    m.temperature = 0.0
    assert m.temperature == 0.0

    with pytest.raises(ValueError):
        m.temperature = 2.5
    with pytest.raises(ValueError):
        QwenModel("qwen-plus", temperature=-0.1)


def test_polymorphic_router_and_dunder_methods():
    router = ModelRouter()
    oai = OpenAIModel("gpt-4o-mini", temperature=0.2)
    qwen = QwenModel("qwen-max", temperature=0.6)

    router.register("oai", oai)
    router.register("qwen", qwen)
    assert len(router) == 2
    assert "qwen" in router

    r1 = router.ask("oai", "测试多态")
    r2 = router.ask("qwen", "测试多态")
    assert r1["provider"] == "openai" and oai.call_count == 1
    assert r2["provider"] == "qwen" and qwen.call_count == 1
    assert "X-DashScope-SSE" in qwen.build_headers()
    assert "openai" in str(oai)

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：扩展 DeepSeekModel 子类

**题目要求：**
继承一个简化的 `SimpleBaseLLM`，实现 `DeepSeekModel` 子类，新增 `enable_thinking: bool` 属性；当 `enable_thinking=True` 时，回复前自动加上 `<think>深度推理中...</think>` 前缀。

**思路提示：** 在子类 `__init__` 中调用 `super().__init__(model_name)` 并保存 `self.enable_thinking`。


In [ ]:
"""Day 9 课后题 1：扩展支持思考模式的 DeepSeekModel 子类。"""


class SimpleBaseLLM:
    def __init__(self, model_name: str) -> None:
        self.model_name = model_name

    def generate(self, prompt: str) -> str:
        raise NotImplementedError


class DeepSeekModel(SimpleBaseLLM):
    def __init__(self, model_name: str = "deepseek-reasoner", enable_thinking: bool = True) -> None:
        super().__init__(model_name)
        self.enable_thinking = enable_thinking

    def generate(self, prompt: str) -> str:
        prefix = "<think>正在拆解问题逻辑...</think> " if self.enable_thinking else ""
        return f"{prefix}[{self.model_name}] 回答: {prompt}"


if __name__ == "__main__":
    ds = DeepSeekModel(enable_thinking=True)
    print(ds.generate("解释什么是多态"))


**答案解析与评分标准：**

考查 `super().__init__` 复用父类初始化以及子类扩展特有行为属性。


### 作业 2（企业实战）：企业实战：带只读余额与充值拦截的 ApiWallet 类

**题目要求：**
设计 `ApiWallet` 类，使用 `@property` 暴露只读余额 `balance`，提供 `deposit(amount)` 与 `charge(amount)` 方法，禁止负数充值、禁止余额透支。

**思路提示：** 底层用 `self._balance` 存储，任何非法金额均抛出 `ValueError`。


In [ ]:
"""Day 9 课后题 2：受保护的企业 API 余额钱包类。"""


class ApiWallet:
    def __init__(self, initial_balance: float = 0.0) -> None:
        if initial_balance < 0:
            raise ValueError("初始余额不能为负数")
        self._balance = float(initial_balance)

    @property
    def balance(self) -> float:
        return round(self._balance, 4)

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("充值金额必须大于 0")
        self._balance += amount

    def charge(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("扣费金额必须大于 0")
        if amount > self._balance:
            raise ValueError(f"余额不足: 当前 {self.balance}，需扣 {amount}")
        self._balance -= amount


if __name__ == "__main__":
    w = ApiWallet(10.0)
    w.deposit(5.5)
    w.charge(3.2)
    print("剩余余额:", w.balance)


**答案解析与评分标准：**

资金与配额类对象绝不允许外部直接赋值 `w.balance = 99999`，必须通过只读 `@property` + 受控方法变更状态。


### 作业 3（架构进阶）：架构进阶：支持上下文管理器（with 语句）的耗时统计器

**题目要求：**
利用魔术方法 `__enter__` 与 `__exit__` 实现 `TimerContext` 类，使调用方可以用 `with TimerContext('LLM调用') as t:` 自动测量代码块执行耗时（毫秒）。

**思路提示：** `__enter__` 记录 `time.perf_counter()` 并返回 `self`；`__exit__` 计算差值存入 `self.elapsed_ms`。


In [ ]:
"""Day 9 课后题 3：基于 __enter__ / __exit__ 的模型调用耗时监控器。"""

import time


class TimerContext:
    def __init__(self, label: str) -> None:
        self.label = label
        self.elapsed_ms: float = 0.0
        self._start: float = 0.0

    def __enter__(self) -> "TimerContext":
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        self.elapsed_ms = round((time.perf_counter() - self._start) * 1000.0, 3)
        # 返回 False 表示不吞掉代码块内部的异常
        return False


if __name__ == "__main__":
    with TimerContext("模拟推理") as timer:
        _ = sum(range(10000))
    print(f"[{timer.label}] 耗时: {timer.elapsed_ms} ms")


**答案解析与评分标准：**

掌握上下文管理器协议是读懂数据库连接池、文件句柄、分布式锁源码的必备技能。


---

## 🎯 老陈收尾复盘

看看咱们刚写完的 `ModelRouter`：不管传进来的是 `OpenAIModel` 还是 `QwenModel`，它根本不关心具体类型，只要都是 `BaseModel` 的子类就能无缝切换，这就是多态（Polymorphism）的威力！明天咱们学习模块化分包、异常处理和虚拟环境，正式向多文件工程迈进。
